# Análise Global de Mudanças Climáticas com Apache Spark


Este notebook processa dados históricos de temperatura global (Berkeley Earth) e de emissões de CO2 (Our World in Data) usando **PySpark** em modo local (`local[*]`), para identificar tendências de aquecimento, anomalias térmicas e a correlação entre emissões e temperatura.


## 1. Ambiente

O ambiente (Java 17, Python 3.11, virtualenv, dependências) é configurado **fora** deste notebook. Siga o passo a passo completo em **`README.md`**.

Resumo:

1. Instalar versões via `mise install` (ou Java 17 + Python 3.11 manualmente).
2. `python -m venv .venv` e ativar o virtualenv.
3. `pip install -r requirements.txt`.
4. Selecionar o kernel **`.venv`** neste notebook.

Verifique antes de continuar: `python teste_spark.py` deve imprimir a versão do Spark e uma tabela de exemplo.

## 2. Datasets

Os CSVs não estão versionados. Baixe e coloque ambos em `data/`:

- **Temperaturas (Berkeley Earth)** — Kaggle: `GlobalLandTemperaturesByCity.csv` (https://www.kaggle.com/datasets/berkeleyearth/climate-change-earth-surface-temperature-data)
- **Emissões de CO2 (Our World in Data)** — `owid-co2-data.csv` (https://github.com/owid/co2-data)

Caminhos esperados: `data/GlobalLandTemperaturesByCity.csv` e `data/owid-co2-data.csv`.

## 3. Inicializar o Spark

`findspark` localiza a instalação do PySpark e a adiciona ao `sys.path` antes do import. A `SparkSession` é o ponto de entrada da API de DataFrames: aqui roda em modo local usando todos os núcleos (`local[*]`), com memória de driver e executor configuradas e o *Adaptive Query Execution* ativado.

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("AnaliseClimatica") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark UI disponível em: http://localhost:4040")

Picked up JAVA_TOOL_OPTIONS: -XX:-UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -XX:-UseContainerSupport
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 13:55:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI disponível em: http://localhost:4040


In [3]:
# Verificação rápida do ambiente Spark
print("Versão do Spark:", spark.version)

_dados = [("Brasil", 27.5), ("Canada", -5.2), ("India", 30.1)]
_df = spark.createDataFrame(_dados, ["Pais", "Temp"])
_df.show()

Versão do Spark: 4.1.1
+------+----+
|  Pais|Temp|
+------+----+
|Brasil|27.5|
|Canada|-5.2|
| India|30.1|
+------+----+



## 4. Próximos passos

Falta fazer:

1. **ETL** — leitura dos CSVs, limpeza (nulos, datas, incerteza, nomes de países), join temperatura × CO2.
2. **As 8 perguntas analíticas** do enunciado (médias móveis, anomalias, correlações, window functions, regressão MLlib).
3. **Visualizações** e artefatos de saída (Parquet).